In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import google.generativeai as genai
import os

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="google.generativeai")

os.environ["GOOGLE_API_KEY"] = "Add Your API Key here"
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

In [2]:
def read_csvfile(filename):
    """Read CSV file with flexible path"""
    path="data"
    filepath=Path(path)
    file = pd.read_csv(filepath/filename, encoding='latin1')
    return file

In [3]:
def clean_column_names(df):
    df=df.copy()
    df.columns = [re.sub(r"[^0-9a-zA-Z]", "_", str(c).strip()).strip("_").lower()
                  for c in df.columns]
    return df

In [4]:
def clean_date_column(df, date_col):
    df=df.copy()
    for d in date_col:
        df[d]=pd.to_datetime(df[d], errors="coerce")
    return df

In [5]:
def filter_data(df):
    df=df.copy()
    df=df[(df["university_campus"]=="Newark") & (~df["exam_workflow_status"].isin(["Cancelled", "Duplicate"]))]
    return df

In [6]:
def date_range_of_data(df, start_date, end_date):
    filtered_data = df[(df["scheduled_date"]>=start_date) & (df["scheduled_date"]<=end_date)]
    exams_per_day = filtered_data.groupby("scheduled_date").size().reset_index(name="exam_count")
    return exams_per_day

Assumption: If number of exams is more than or equal to 15, it's considered as peak exam day

In [8]:
def find_peak_exam_days(df, start_date, end_date):
    exams_per_day = date_range_of_data(df, start_date, end_date)
    peak_days = exams_per_day.sort_values("exam_count", ascending=False)
    peak_days = peak_days[peak_days["exam_count"]>=15]
    return peak_days                                                        

In [9]:
def find_peak_exam_weekday(df, start_date, end_date):
    exams_per_day = date_range_of_data(df, start_date, end_date)       
    exams_per_day["day_of_the_week"] = exams_per_day["scheduled_date"].dt.day_name()
    peak_day_of_the_week = exams_per_day.groupby("day_of_the_week").sum("exam_count").sort_values("exam_count", ascending=False)
    return peak_day_of_the_week  

In [10]:
def find_peak_exam_month(df, start_date, end_date):
    exams_per_day = date_range_of_data(df, start_date, end_date)       
    exams_per_day["month_of_year"] = exams_per_day["scheduled_date"].dt.month_name()
    peak_month = exams_per_day.groupby("month_of_year").sum("exam_count").sort_values("exam_count", ascending=False)
    return peak_month

Assumption: Each classroom can accommodate 7 students with special needs

In [12]:
def find_rooms_required(df, start_date, end_date):
    peak_exam_days = find_peak_exam_days(df, start_date, end_date)
    peak_exam_days["rooms_required"]= np.ceil(peak_exam_days["exam_count"]/7).astype(int)
    return peak_exam_days

Assumption: Each classroom needs 2 proctors at a time

In [14]:
def find_number_of_proctors(df, start_date, end_date):
    """
    Find number of proctors needed for the day assuming each classroom needs 2 proctors at a time
    parameters: dataframe, start date and end date of the date range you want the data of
    returns: dataframe with an additional column containing count of proctors needed
    """
    peak_days = find_rooms_required(df, start_date, end_date)
    peak_days["proctors_required"] = peak_days["rooms_required"]*2
    return peak_days

In [ ]:
print("Welcome to Exam AI Assistant")
print("=" * 60)
csv_file = input("Enter CSV file name: ").strip()
if not csv_file:
    csv_file = "sample_data.csv"

print(f"\nLoading {csv_file}...")
df = read_csvfile(csv_file)
df = clean_column_names(df)
df = clean_date_column(df, ["scheduled_date"])
df = filter_data(df)

print(f"Loaded {len(df)} exams successfully")
print(f"Date range: {df['scheduled_date'].min().date()} to {df['scheduled_date'].max().date()}")

# Get date range for analysis
print(f"\nPlease enter the date range for analysis")
start_date = input("\nStart date (YYYY-MM-DD): ").strip()
end_date = input("End date (YYYY-MM-DD): ").strip()

Welcome to Exam AI Assistant


In [ ]:
def gather_exam_statistics(df, start_date, end_date):
    """Gathers all exam data in one place"""
    
    exam_data = {
        "monthly": find_peak_exam_month(df, start_date, end_date),
        "peak_days": find_peak_exam_days(df, start_date, end_date),
        "weekdays": find_peak_exam_weekday(df, start_date, end_date),
        "proctors": find_number_of_proctors(df, start_date, end_date),
        "rooms": find_rooms_required(df, start_date, end_date),
    }
    
    return exam_data

In [ ]:
def format_monthly_data(monthly_data):
    """Format monthly statistics as readable string"""
    return "\n".join([f"  {month}: {count} exams" for month, count in monthly_data.items()])


def format_peak_days(peak_days):
    """Format peak days as readable string"""
    return "\n".join([f"  {day.date()}: {count} exams" 
                      for day, count in peak_days[['scheduled_date', 'exam_count']].values])


def format_weekday_data(weekday_data):
    """Format weekday statistics as readable string"""
    return "\n".join([f"  {day}: {count} exams" for day, count in weekday_data.items()])


def format_resource_requirements(proctors_data, rooms_data):
    """Format resource requirements as readable string"""
    max_proctors = proctors_data["proctors_required"].max()
    max_rooms = rooms_data["rooms_required"].max()
    
    return f"""  - Max Proctors Needed: {int(max_proctors)}
    - Max Rooms Needed: {int(max_rooms)}"""


In [ ]:
def build_gemini_prompt(question, statistics, start_date, end_date):
    """Build the prompt to send to Gemini AI"""
    
    # Format all exam related data
    monthly_str = format_monthly_data(statistics["monthly"])
    peak_days_str = format_peak_days(statistics["peak_days"])
    weekday_str = format_weekday_data(statistics["weekdays"])
    resources_str = format_resource_requirements(statistics["proctors"], statistics["rooms"])
    
    # Build the prompt
    prompt = f"""You are a Data Analyst. Use the following data context to answer the user's question.

ACTUAL DATA ANALYSIS RESULTS ({start_date} to {end_date}):

MONTHLY BREAKDOWN:
{monthly_str}

PEAK DAYS:
{peak_days_str}

WEEKDAY DISTRIBUTION:
{weekday_str}

RESOURCE REQUIREMENTS:
{resources_str}

USER QUESTION: {question}

Based on the ACTUAL DATA shown above, answer the user's question with specific numbers and recommendations."""
    
    return prompt


In [ ]:
def ask_gemini(prompt):
    """Send prompt to Gemini and get response"""
    model = genai.GenerativeModel('gemini-flash-latest')
    response = model.generate_content(prompt)
    return response.text

In [ ]:
def ask_gemini_about_exams(question, df, start_date, end_date):
    """Main function that coordinates all the above steps."""
    
    # Step 1: Gather statistics
    statistics = gather_exam_statistics(df, start_date, end_date)
    
    # Step 2: Build prompt
    prompt = build_gemini_prompt(question, statistics, start_date, end_date)
    
    # Step 3: Get answer from Gemini
    answer = ask_gemini(prompt)
    
    # Step 4: Return answer
    return answer

In [ ]:
def run_interactive_chat(df, start_date, end_date):
    """Run the interactive Q&A loop"""
    
    print("\n" + "=" * 60)
    print("Ask questions about your exam data!")
    print("Type 'quit' to exit")
    print("=" * 60 + "\n")
    
    while True:
        question = input("Your question: ").strip()
        
        # Check for exit
        if question.lower() in ['quit', 'exit', 'stop']:
            print("\nExiting.....")
            break
        
        # Check for empty input
        if not question:
            print("Please enter a question.\n")
            continue
        
        # Get and display answer
        print("\nGemini is analyzing your data...\n")
        
        try:
            answer = ask_gemini_about_exams(question, df, start_date, end_date)
            print(f"Answer:\n{answer}\n")
            print("-" * 60 + "\n")
        except Exception as e:
            print(f"Error: {str(e)}")
            print("Make sure your API_KEY is set.\n")

run_interactive_chat(df, start_date, end_date)